Files originally from https://github.com/LucasSilvaFerreira/Perturb_Loader

In [1]:
import mudata as md
import anndata as ad
import numpy as np
import pandas as pd
import perturbvi
import os
import pyro

smoke_test = True


In [ ]:
def load_adata(data_dir = "."):
    rna_adata = ad.read_h5ad(f"{data_dir}/ann_exp.h5ad")
    rna_adata.obs['library_size']=rna_adata.X.sum(axis=1)
    rna_adata.varm['gene_tested'] = ad.read_h5ad(f"{data_dir}/ann_Element_x_tested_genes.h5ad").to_df().T
    grna_adata = ad.read_h5ad(f"{data_dir}/ann_guide.h5ad")
    grna_adata.varm['gene_targeted'] = ad.read_h5ad(f"{data_dir}/ann_Element_guide.h5ad").to_df().T
    mdata = md.MuData({'rna':rna_adata, 'grna':grna_adata})
    mdata.write_h5mu(f'{data_dir}/gasperini_pilot_highMOI.h5mu')
    return mdata

force = True
mudata_file = "gasperini_pilot_highMOI.h5mu"
data_dir = "../../../../Data/gasperini_pilot"
if mudata_file not in os.listdir(data_dir) or force:
    mdata = load_adata(data_dir)
else:
    mdata = md.read_h5mu(os.path.join(data_dir, mudata_file))

In [ ]:
mdata['grna']

AnnData object with n_obs × n_vars = 47964 × 3115
    obs: 'bath_number', 'percent_mito', 'log_number_of_detected_genes', 'log_total_gene_count', 'log_total_guide_count'
    varm: 'gene_targeted'

In [ ]:
grna_subset = mdata['grna'][:,mdata['grna'].var_names.str.contains(r"TSS|random|scrambled", regex=True)]
genes = {e.split('_')[0] for e in grna_subset.var_names if '_TSS' in e}
subset_genes = [g for g in genes if g in mdata['rna'].var_names]

In [ ]:
# subset down to only control perturbed genes
# rna_subset = mdata['rna'][:,mdata['rna'].varm['gene_tested'].sum(axis=1).values > 0]
if smoke_test:
    subset_genes = subset_genes[:10]
rna_subset =  mdata['rna'][:,subset_genes]
mdata_subset = md.MuData({'rna':rna_subset.copy(), 'grna': grna_subset.copy()})
mdata_subset

/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/mudata/_core/mudata.py:446: UserWarning: var_names are not unique. To make them unique, call `.var_names_make_unique`.
  warnings.warn(


MuData object with n_obs × n_vars = 47964 × 872
  2 modalities
    rna:	47964 x 10
      obs:	'bath_number', 'percent_mito', 'log_number_of_detected_genes', 'log_total_gene_count', 'log_total_guide_count', 'library_size'
      varm:	'gene_tested'
    grna:	47964 x 862
      obs:	'bath_number', 'percent_mito', 'log_number_of_detected_genes', 'log_total_gene_count', 'log_total_guide_count'
      varm:	'gene_targeted'

In [ ]:
mdata['rna'].var_names[:100]

Index(['AL627309.1', 'AL627309.5', 'AP006222.1', 'AL732372.2', 'AL669831.3',
       'MTND1P23', 'MTND2P28', 'MTCO1P12', 'AC114498.2', 'MTATP6P1',
       'AL669831.1', 'LINC01409', 'LINC01128', 'LINC00115', 'AL645608.2',
       'NOC2L', 'KLHL17', 'PLEKHN1', 'HES4', 'ISG15', 'AGRN', 'AL390719.1',
       'RNF223', 'C1orf159', 'TTLL10', 'SDF4', 'B3GALT6', 'C1QTNF12', 'UBE2J2',
       'ACAP3', 'PUSL1', 'INTS11', 'CPTP', 'DVL1', 'MXRA8', 'AURKAIP1',
       'CCNL2', 'MRPL20-AS1', 'MRPL20', 'AL391244.2', 'VWA1', 'ATAD3C',
       'ATAD3B', 'ATAD3A', 'SSU72', 'AL645728.1', 'FNDC10', 'AL691432.2',
       'MIB2', 'MMP23B', 'CDK11B', 'FO704657.1', 'SLC35E2B', 'CDK11A',
       'AL031282.1', 'SLC35E2A', 'NADK', 'GNB1', 'TMEM52', 'CFAP74',
       'AL391845.2', 'GABRD', 'PRKCZ', 'AL590822.2', 'FAAP20', 'SKI', 'MORN1',
       'RER1', 'PEX10', 'PANK4', 'AL139246.5', 'TNFRSF14-AS1', 'TNFRSF14',
       'PRXL2B', 'AL512383.1', 'TPRG1L', 'WRAP73', 'TP73', 'TP73-AS1',
       'CCDC27', 'SMIM1', 'LRRC47', 'CEP1

In [ ]:
perturbvi.PERTURBVI.setup_mudata(
    mdata_subset,
    batch_key="bath_number",
    size_factor_key="library_size",
    modalities={
        "rna_layer": 'rna',
        "perturbation_layer": 'grna',
    },
)

model = perturbvi.PERTURBVI(mdata_subset)
model.view_anndata_setup()

Anndata setup with scvi-tools version 0.20.1.

Setup via `PERTURBVI.setup_anndata` with arguments:

{
│   'rna_layer': None,
│   'batch_key': 'bath_number',
│   'perturbation_layer': None,
│   'modalities': {'rna_layer': 'rna', 'perturbation_layer': 'grna'},
│   'size_factor_key': 'library_size'
}

     Summary Statistics     
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Summary Stat Key ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│     n_batch      │   6   │
│     n_cells      │ 47964 │
│ n_perturbations  │  862  │
│      n_vars      │  10   │
└──────────────────┴───────┘

                       Data Registry                        
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   Registry Key    ┃         scvi-tools Location          ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         X         │          adata.mod['rna'].X          │
│       batch       │ adata.mod['rna'].obs['_scvi_batch']  │
│       ind_x       │    adata.mod['rna'].obs['_ind_x']    │
│ observed_lib_size │ adata.mod['rna'].obs['library_size'] │
│   perturbations   │         adata.mod['grna'].X          │
└───────────────────┴──────────────────────────────────────┘

                     batch State Registry                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃     Source Location      ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['bath_number'] │     1      │          0          │
│                          │     2      │          1          │
│                          │     3      │          2          │
│                          │     4      │          3          │
│                          │     5      │          4          │
│                          │     6      │          5          │
└──────────────────────────┴────────────┴─────────────────────┘

In [ ]:
# optimizer = pyro.optim.ClippedAdam({'lr':0.001, 'lrd':0.9})
model.train(
    max_epochs=20,
    train_size=1,
    batch_size=1024,
    lr=0.1
    # plan_kwargs={"optim": optimizer},
)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/configuration_validator.py:106: UserWarning: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
  rank_zero_warn("You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.")


Epoch 20/20: 100%|██████████| 20/20 [00:10<00:00,  1.94it/s, v_num=1, elbo_train=1.22e+6]

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 20/20: 100%|██████████| 20/20 [00:10<00:00,  1.90it/s, v_num=1, elbo_train=1.22e+6]


In [ ]:
# optimizer = pyro.optim.ClippedAdam({'lr':0.001, 'lrd':0.9})
model.train(
    max_epochs=50,
    train_size=1,
    batch_size=1024,
    lr=0.001
    # plan_kwargs={"optim": optimizer},
)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/configuration_validator.py:106: UserWarning: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
  rank_zero_warn("You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.")


Epoch 50/50: 100%|██████████| 50/50 [00:24<00:00,  2.11it/s, v_num=1, elbo_train=1.13e+6]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 50/50: 100%|██████████| 50/50 [00:24<00:00,  2.03it/s, v_num=1, elbo_train=1.13e+6]


In [ ]:
%load_ext autoreload
%autoreload 2
from scipy.stats import norm

if smoke_test:
    for gene_index in range(len(subset_genes)):
        print(mdata_subset['rna'].var_names[gene_index])

        perturb_mean_lfc_mu = pyro.get_param_store()['perturb_mean_lfc.mu'].detach().cpu().numpy()
        perturb_disp_lfc_mu = pyro.get_param_store()['perturb_disp_lfc.mu'].detach().cpu().numpy()
        lfc_cov_tril = pyro.get_param_store()['perturb_lfc.scale_tril'] * pyro.get_param_store()['scale_factor'].exp()
        lfc_cov = lfc_cov_tril @ lfc_cov_tril.transpose(dim0=-1, dim1=-2)
        perturb_mean_lfc_sigma = lfc_cov[...,0,0].sqrt().detach().cpu().numpy()
        perturb_disp_lfc_sigma = lfc_cov[...,0,0].sqrt().detach().cpu().numpy()
        assert perturb_mean_lfc_mu.shape == perturb_mean_lfc_sigma.shape
        perturb_z_scores = perturb_mean_lfc_mu/perturb_mean_lfc_sigma
        perturb_p_vals = norm.cdf(0, loc=-perturb_mean_lfc_mu, scale=perturb_mean_lfc_sigma)

        # perturb_z_scores[:,gene_index].detach().cpu().numpy()
        z_df = pd.DataFrame({'z_score': perturb_z_scores[:,gene_index],
                            'mean_mu': perturb_mean_lfc_mu[:,gene_index],
                            'disp_mu': perturb_disp_lfc_mu[:,gene_index],
                            'mu_p_val': perturb_p_vals[:,gene_index],
                            'grna': mdata_subset['grna'].var_names,})
        # z_df.hist('mu_p_val', bins=100)
        # z_df.plot(x='mean_mu',y='disp_mu', style='o')
        print(z_df.sort_values('mu_p_val', ascending=True).head(20))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
ACTB
      z_score   mean_mu   disp_mu  mu_p_val            grna
1   -4.140476 -0.213915  0.115851  0.000017      ACTB_TSS|2
530 -1.362011 -0.070368  0.073903  0.086597      RBMX_TSS|1
178 -1.238527 -0.063988  0.065730  0.107760     EIF3A_TSS|1
525 -1.112714 -0.057488  0.045793  0.132916       RAN_TSS|2
504 -1.018786 -0.052635  0.054628  0.154152     PSMB6_TSS|1
707 -1.012977 -0.052335  0.045067  0.155535      TYMS_TSS|2
334 -0.991570 -0.051229  0.047486  0.160704      MCM3_TSS|1
264 -0.938312 -0.048477  0.043795  0.174042     HMGB3_TSS|1
586 -0.921716 -0.047620  0.042567  0.178338      SAT1_TSS|1
185 -0.916783 -0.047365  0.043218  0.179628     EIF3J_TSS|2
445 -0.881709 -0.045553  0.049510  0.188967     PAICS_TSS|2
152 -0.867877 -0.044838  0.037913  0.192731     DDX3X_TSS|1
238 -0.846230 -0.043720  0.047442  0.198712      GNB2_TSS|1
197 -0.841119 -0.043456  0.037825  0.200141    ELAVL1_TSS|2
548 -0.

In [ ]:
for k, v in pyro.get_param_store().items():
    print (k, v.shape)

scale_factor torch.Size([])
log_var_mean.mu torch.Size([10])
log_var_disp.mu torch.Size([10])
batch_effect.mu torch.Size([6, 1])
batch_effect.sigma torch.Size([6, 1])
log_var_mean.sigma torch.Size([10])
log_var_disp.sigma torch.Size([10])
perturb_mean_lfc.mu torch.Size([862, 10])
perturb_disp_lfc.mu torch.Size([862, 10])
perturb_lfc.scale_tril torch.Size([862, 10, 2, 2])
